# OCCTO ユニット別発電実績 分析ノートブック

`dlh_dev` カタログの `silver.occto_unit_generation_actuals` を PyIceberg 経由で読み込み、エリア別・発電方式別・発電所別の切り口で発電実績を可視化する。

テーブルは「発電所 × ユニット × 対象日 × 30分コマ（`time_code` 1〜48, 開始時刻基準）」のlong 粒度。詳細は `docs/tasks/plan_occto_pipeline.md` を参照。

## 1. セットアップ

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import plotly.express as px
import polars as pl

WORKSPACE_ROOT = (
    Path.cwd().resolve().parents[1] if Path.cwd().name == "Jupyter" else Path("/workspace")
)
SRC_PATH = WORKSPACE_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from common.iceberg import get_catalog

pl.Config.set_tbl_rows(20)

polars.config.Config

## 2. カタログへの接続とテーブル読み込み

`configuration/iceberg/.pyiceberg.yaml` の `dlh_dev` カタログ（SQLite カタログ / RustFS ウェアハウス）に接続し、OCCTO silver テーブルを Polars DataFrame として読み込む。

In [2]:
TABLE_IDENTIFIER = "silver.occto_unit_generation_actuals"

catalog = get_catalog("dlh_dev")
df = catalog.load_table(TABLE_IDENTIFIER).scan().to_polars()
print(f"shape={df.shape}")

shape=(19681872, 15)


## 3. スキーマと期間の確認

In [3]:
df.schema

Schema([('power_plant_code', String),
        ('unit_name', String),
        ('target_date', Date),
        ('time_code', Int32),
        ('delivery_datetime', Datetime(time_unit='us', time_zone='UTC')),
        ('generation_kwh', Int64),
        ('area', String),
        ('power_plant_name', String),
        ('power_generation_method_and_fuel_type', String),
        ('updated_datetime', Datetime(time_unit='us', time_zone='UTC')),
        ('source_data', String),
        ('status', String),
        ('ingestion_time', Datetime(time_unit='us', time_zone='UTC')),
        ('ingestion_date', Date),
        ('execution_id', String)])

In [4]:
date_range = df.select(
    pl.col("target_date").min().alias("min_target_date"),
    pl.col("target_date").max().alias("max_target_date"),
)
date_range

min_target_date,max_target_date
date,date
2024-03-25,2026-08-09


## 4. データ品質チェック

欠損値の件数と、発電所・ユニット・エリア・発電方式の件数を確認する。`generation_kwh` の欠損は、DuckDB の `UNPIVOT` 段階で「その日その発電所のコマの一部だけが欠測」だったケースに相当する（詳細は `bronze_to_silver_occto_unit_generation_actuals.py` 参照）。

In [5]:
df.null_count()

power_plant_code,unit_name,target_date,time_code,delivery_datetime,generation_kwh,area,power_plant_name,power_generation_method_and_fuel_type,updated_datetime,source_data,status,ingestion_time,ingestion_date,execution_id
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,3971,0,0,208320,0,0,0,0,0,0


In [6]:
n_plants = df["power_plant_code"].n_unique()
n_units = df.select(["power_plant_code", "unit_name"]).unique().height
n_areas = df["area"].n_unique()
n_methods = df["power_generation_method_and_fuel_type"].n_unique()

print(f"発電所数: {n_plants}")
print(f"発電所×ユニット数: {n_units}")
print(f"エリア数: {n_areas}")
print(f"発電方式数: {n_methods}")

発電所数: 301
発電所×ユニット数: 491
エリア数: 10
発電方式数: 7


## 5. エリア別 総発電量

対象期間全体の `generation_kwh` をエリアごとに合計する（GWh 換算）。

In [7]:
area_total = (
    df.group_by("area")
    .agg((pl.col("generation_kwh").sum() / 1e6).alias("total_gwh"))
    .sort("total_gwh", descending=True)
)

fig = px.bar(
    area_total,
    x="area",
    y="total_gwh",
    title="エリア別 総発電量",
    labels={"area": "エリア", "total_gwh": "総発電量 (GWh)"},
)
fig.show()

## 6. 発電方式別 総発電量

`power_generation_method_and_fuel_type` ごとに合計する。

In [8]:
method_total = (
    df.group_by("power_generation_method_and_fuel_type")
    .agg((pl.col("generation_kwh").sum() / 1e6).alias("total_gwh"))
    .sort("total_gwh", descending=True)
)

fig = px.bar(
    method_total,
    x="power_generation_method_and_fuel_type",
    y="total_gwh",
    title="発電方式別 総発電量",
    labels={
        "power_generation_method_and_fuel_type": "発電方式・燃種",
        "total_gwh": "総発電量 (GWh)",
    },
)
fig.show()

## 7. 発電所別 総発電量ランキング（上位20）

発電所は全国で301箇所あるため、まず上位20箇所を横棒グラフで比較する。以降のセルはここで作るランキングを土台にする。

In [9]:
TOP_N = 20

plant_ranking = (
    df.group_by(["power_plant_code", "power_plant_name", "area"])
    .agg((pl.col("generation_kwh").sum() / 1e6).alias("total_gwh"))
    .sort("total_gwh", descending=True)
)

top_plants = plant_ranking.head(TOP_N)

fig = px.bar(
    top_plants.sort("total_gwh"),
    x="total_gwh",
    y="power_plant_name",
    color="area",
    orientation="h",
    title=f"発電所別 総発電量ランキング（上位{TOP_N}）",
    labels={
        "power_plant_name": "発電所名",
        "total_gwh": "総発電量 (GWh)",
        "area": "エリア",
    },
    height=600,
)
fig.show()

## 8. 日次総発電量の推移（全国合計）

全発電所を合計した日次発電量の時系列。データの欠落や異常値がないかの全体感の確認も兼ねる。

In [10]:
daily_total = (
    df.group_by("target_date")
    .agg((pl.col("generation_kwh").sum() / 1e6).alias("total_gwh"))
    .sort("target_date")
)

fig = px.line(
    daily_total,
    x="target_date",
    y="total_gwh",
    title="日次総発電量の推移（全国合計）",
    labels={"target_date": "対象日", "total_gwh": "総発電量 (GWh)"},
)
fig.show()

## 9. 上位発電所の日次発電量推移

セクション7で求めた上位5発電所について、日次発電量の推移を重ねて比較する。

In [11]:
TOP_N_TREND = 5
top_plant_codes = top_plants.head(TOP_N_TREND)["power_plant_code"].to_list()

daily_by_top_plant = (
    df.filter(pl.col("power_plant_code").is_in(top_plant_codes))
    .group_by(["target_date", "power_plant_name"])
    .agg((pl.col("generation_kwh").sum() / 1e3).alias("total_mwh"))
    .sort("target_date")
)

fig = px.line(
    daily_by_top_plant,
    x="target_date",
    y="total_mwh",
    color="power_plant_name",
    title=f"上位{TOP_N_TREND}発電所の日次発電量推移",
    labels={
        "target_date": "対象日",
        "total_mwh": "日次発電量 (MWh)",
        "power_plant_name": "発電所名",
    },
)
fig.show()

## 10. 発電所別 1日の発電カーブ（コマ別平均）

上位3発電所について、`time_code`（30分コマ、開始時刻基準）ごとに全期間の平均発電量を求め、典型的な1日の発電パターンを比較する。原子力・石炭火力は日内でほぼ一定（ベースロード）になりやすく、水力などは変動が大きくなりやすい。

In [12]:
TOP_N_PROFILE = 3
profile_plant_codes = top_plants.head(TOP_N_PROFILE)["power_plant_code"].to_list()


def slot_label(time_code: int) -> str:
    """time_code は開始時刻基準（1 -> 00:00, 48 -> 23:30）。"""
    total_minutes = (time_code - 1) * 30
    hour, minute = divmod(total_minutes, 60)
    return f"{hour % 24:02d}:{minute:02d}"


SLOT_ORDER = [slot_label(tc) for tc in range(1, 49)]

intraday_profile = (
    df.filter(pl.col("power_plant_code").is_in(profile_plant_codes))
    .group_by(["power_plant_name", "time_code"])
    .agg(pl.col("generation_kwh").mean().alias("avg_generation_kwh"))
    .sort(["power_plant_name", "time_code"])
    .with_columns(
        pl.col("time_code")
        .map_elements(slot_label, return_dtype=pl.Utf8)
        .alias("slot_start")
    )
)

fig = px.line(
    intraday_profile,
    x="slot_start",
    y="avg_generation_kwh",
    color="power_plant_name",
    category_orders={"slot_start": SLOT_ORDER},
    title="発電所別 1日の発電カーブ（コマ別平均, 全期間平均）",
    labels={
        "slot_start": "時刻（コマ開始時刻）",
        "avg_generation_kwh": "平均発電量 (kWh/30分)",
        "power_plant_name": "発電所名",
    },
)
fig.show()

## 11. 個別発電所の詳細確認

`SELECTED_PLANT_CODE` を変更すれば任意の発電所（全301箇所）の実績を確認できる。既定値はセクション7の1位（総発電量が最大の発電所）。

In [13]:
# 発電所コードを変更すると、その発電所の実績に切り替わる
SELECTED_PLANT_CODE = top_plants[0, "power_plant_code"]

plant_df = df.filter(pl.col("power_plant_code") == SELECTED_PLANT_CODE)
plant_name = plant_df[0, "power_plant_name"]
plant_area = plant_df[0, "area"]
plant_method = plant_df[0, "power_generation_method_and_fuel_type"]

print(f"発電所: {plant_name}（{SELECTED_PLANT_CODE}）")
print(f"エリア: {plant_area} / 発電方式: {plant_method}")

発電所: 玄海原子力発電所（95001）
エリア: 九州 / 発電方式: 原子力


In [14]:
plant_daily = (
    plant_df.group_by("target_date")
    .agg((pl.col("generation_kwh").sum() / 1e3).alias("total_mwh"))
    .sort("target_date")
)

fig = px.line(
    plant_daily,
    x="target_date",
    y="total_mwh",
    title=f"{plant_name} の日次発電量推移",
    labels={"target_date": "対象日", "total_mwh": "日次発電量 (MWh)"},
)
fig.show()

In [15]:
plant_intraday = (
    plant_df.group_by("time_code")
    .agg(
        pl.col("generation_kwh").mean().alias("avg_generation_kwh"),
        pl.col("generation_kwh").std().alias("std_generation_kwh"),
    )
    .sort("time_code")
    .with_columns(
        pl.col("time_code")
        .map_elements(slot_label, return_dtype=pl.Utf8)
        .alias("slot_start")
    )
)

fig = px.line(
    plant_intraday,
    x="slot_start",
    y="avg_generation_kwh",
    error_y="std_generation_kwh",
    category_orders={"slot_start": SLOT_ORDER},
    title=f"{plant_name} の1日の発電カーブ（コマ別平均 ± 標準偏差）",
    labels={
        "slot_start": "時刻（コマ開始時刻）",
        "avg_generation_kwh": "平均発電量 (kWh/30分)",
    },
)
fig.show()